In [ ]:
from astropy.coordinates import SkyCoord
from astropy import units as u
import glob
import numpy as np
from astropy.table import Table
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
from matplotlib import gridspec
import datetime
import os
from astropy.io import fits
from astropy import wcs
filtername='F560W'
skycoord = SkyCoord(ra="19:23:41.1829762303", dec="14:30:34.6309739969", unit=(u.hourangle, u.deg), frame='icrs')
        
def load_data(filename):
    fh = fits.open(filename)
    im1 = fh
    data = im1['SCI'].data
    try:
        wht = im1['WHT'].data
    except KeyError:
        wht = None
    err = im1['ERR'].data
    instrument = im1[0].header['INSTRUME']
    telescope = im1[0].header['TELESCOP']
    obsdate = im1[0].header['DATE-OBS']
    return fh, im1, data, wht, err, instrument, telescope, obsdate    

def get_filenames(basepath, filtername, proposal_id, field, each_suffix, module, pupil='clear', visitid='001'):

    # jw01182004002_02101_00012_nrcalong_destreak_o004_crf.fits
    # jw02221001001_07101_00012_nrcalong_destreak_o001_crf.fits
    # jw02221001001_05101_00022_nrcb3_destreak_o001_crf.fits
        #jw06151002001_02101_00001_mirimage_i2d.fits

    glstr = f'{basepath}/{filtername}/pipeline/jw0{proposal_id}{field}*{module}*_{each_suffix}.fits'
    
  
    fglob = glob.glob(glstr)
    for st in fglob:
        print(st)
        if 'align' in st or 'uncal' in st:
            print(f"Removing {st} from glob string because it is an alignment file")
            fglob.remove(st)
    if len(fglob) == 0:
        raise ValueError(f"No matches found to {glstr}")
    else:
        return fglob
proposal_id = '6151'
target = 'w51_miri'
nvisits = {'2221': {'brick': 1, 'cloudc': 2},
               '1182': {'brick': 2},
               '6151': {'w51': 1, 'w51_miri': 2}
               }
field_to_reg_mapping = {'2221': {'001': 'brick', '002': 'cloudc'},
                        '1182': {'004': 'brick'},
                        '6151': {'001': 'w51', '002':'w51_miri'}}[proposal_id]
reg_to_field_mapping = {v:k for k,v in field_to_reg_mapping.items()}
field = reg_to_field_mapping[target]
target_dir = target
if proposal_id == '6151' and target == 'w51_miri':
    target_dir = 'w51'
basepath = f'/orange/adamginsburg/jwst/{target_dir}/'


modules = ['mirimage']
wl = float(filtername[1:-1]) 

fig = plt.figure(figsize=(18,12))
# make 3 columns, and the number of row will be detrmined by the number of frames
gs = gridspec.GridSpec(4,6, wspace=0.1, hspace=0.1) 
ax1 = fig.add_subplot(gs[0,0])
ax2 = fig.add_subplot(gs[0,1])
ax3 = fig.add_subplot(gs[0,2])
ax4 = fig.add_subplot(gs[0,3])
ax5 = fig.add_subplot(gs[0,4])
ax6 = fig.add_subplot(gs[0,5])
ax7 = fig.add_subplot(gs[1,0])
ax8 = fig.add_subplot(gs[1,1])
ax9 = fig.add_subplot(gs[1,2])
ax10 = fig.add_subplot(gs[1,3])
ax11 = fig.add_subplot(gs[1,4])
ax12 = fig.add_subplot(gs[1,5])
ax13 = fig.add_subplot(gs[2,0])
ax14 = fig.add_subplot(gs[2,1])
ax15 = fig.add_subplot(gs[2,2])
ax16 = fig.add_subplot(gs[2,3])
ax17 = fig.add_subplot(gs[2,4])
ax18 = fig.add_subplot(gs[2,5])
ax19 = fig.add_subplot(gs[3,0])
ax20 = fig.add_subplot(gs[3,1])
ax21 = fig.add_subplot(gs[3,2])
ax22 = fig.add_subplot(gs[3,3])
ax23 = fig.add_subplot(gs[3,4])
ax24 = fig.add_subplot(gs[3,5])
axlist = [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8, ax9, ax10, ax11, ax12, ax13, ax14, ax15, ax16, ax17, ax18, ax19, ax20, ax21, ax22, ax23, ax24]

tblfns_merged = glob.glob(f'/orange/adamginsburg/jwst/w51/catalogs/{filtername.lower()}_*_indivexp_merged_dao_after_merger_combined_with_satstars_fixed.fits')
print('last modified:', datetime.datetime.fromtimestamp(os.path.getmtime(tblfns_merged[0])))
tbls_merged = [Table.read(tblfn) for tblfn in tblfns_merged]
skycoord_merged = tbls_merged[0]['skycoord']

# Match the target coordinate against the catalog
i2d, d2d, _ = skycoord.match_to_catalog_sky(skycoord_merged, nthneighbor=1)
print('closest match:', skycoord_merged[i2d], d2d)

nmatch = tbls_merged[0]['nmatch'][i2d]
print('nmatch', nmatch)
print('nmatch', nmatch)
nmatch_max = 0                             

for module in modules:
    detector = module # no sub-detectors for long-NIRCAM
    
    for ii, visitid in enumerate(range(1, nvisits[proposal_id][target] + 1)):
        if True:
            visitid = f'{visitid}'

            filenames = get_filenames(basepath, filtername, proposal_id,
                                                    field, visitid=visitid,
                                                    each_suffix='cal',
                                                    module=module, pupil='clear') 
            
              
            for ii, filename in enumerate(filenames):
                print('filename, ', filename)
                vgroup = filename.split('/')[-1].split('_')[1]
                exp = filename.split('/')[-1].split('_')[2]
                #f140m_nrca4_visit001_vgroup03109_exp00008_daophot_daofind.fits
                visitid_ = f'{int(visitid):03d}'                 
                daofind_basic_catname = Table.read(f'{basepath}/{filtername.upper()}/{filtername.lower()}_{module}_visit{visitid_}_vgroup{vgroup}_exp{exp}_daophot_daofind.fits')
                star_cat = Table(daofind_basic_catname)


                fh, im1, data, wht, err, instrument, telescope, obsdate = load_data(filename)
            
                
                img = im1['SCI'].data
                # set up coordinate system
                ww = wcs.WCS(im1[1].header)
                pixscale = ww.proj_plane_pixel_area()**0.5
                cen = ww.pixel_to_world(im1[1].shape[1]/2, im1[1].shape[0]/2)

                pixcoord = ww.world_to_pixel(skycoord)

                if (0 <= pixcoord[0] < img.shape[1]) and (0 <= pixcoord[1] < img.shape[0]):
                    print(f"Skycoord {skycoord} is in the FOV of the image {filename}")
                    nmatch_max += 1
                else:
                    print(f"Skycoord {skycoord} is not in the FOV of the image {filename}")
                    continue
                
                # get the cutout of the image around the skycoord
                cutout = Cutout2D(
                    img,
                    position=skycoord,
                    size=(1, 1) * u.arcsec,
                    wcs=ww,
                    mode='partial',
                    fill_value=np.nan,
                )

                cutout_data = cutout.data
                finite = np.isfinite(cutout_data)

                

                cutout_data = cutout_data.copy()
                cutout_data[~finite] = 0
                norm = simple_norm(cutout_data, stretch='log', percent=99)

                axlist[ii].imshow(cutout_data, origin='lower', cmap='gray', norm=norm)
                axlist[ii].scatter(star_cat['xcentroid']-cutout.xmin_original, star_cat['ycentroid']-cutout.ymin_original, s=50, edgecolor='red', facecolor='none')
                axlist[ii].set_title(f'{filename.split("/")[-1]}')
                axlist[ii].set_xlim(0, cutout_data.shape[1])
                axlist[ii].set_ylim(0, cutout_data.shape[0])
print('nmatch_max, ', nmatch_max)

            







